In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, log_loss, 
    classification_report, confusion_matrix
)
from CartClassifier import CARTClassifier

#Random Forest Implementation:


class RandomForestFromScratch:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.trees = []

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        # Sampling with replacement
        indices = np.random.choice(n_samples, n_samples, replace=True)

        return X.iloc[indices].values, y[indices]

    def fit(self, X, y):
        self.trees = []
        for i in range(self.n_estimators):
            X_sample, y_sample = self._bootstrap_sample(X, y)
            
            # Initialize CART
            tree = CARTClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf
            )
            
            # Train the individual tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict_proba(self, X):
        #Calculates probabilities for AUROC, AUPRC, and Log-Loss
        # Convert X to values if it's a DataFrame
        X_vals = X.values if isinstance(X, pd.DataFrame) else X
        
        # Collect predictions (0 or 1) from every tree
        # Shape: (n_estimators, n_samples)
        all_tree_preds = np.array([tree.predict(X_vals) for tree in self.trees])
        
        # The probability of Class 1 is the average of tree votes
        prob_pos = np.mean(all_tree_preds, axis=0)
        
        # Return format: [prob_of_0, prob_of_1]
        return np.vstack([1 - prob_pos, prob_pos]).T

    def predict(self, X):
        #Hard label prediction via majority vote
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

def run_performance_assessment(model, X, y, set_name="Test"):
   # Evaluates Accuracy, Precision, Recall, F1, Log-Loss, AUROC, and AUPRC.
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1] # Probability of the positive (immunogenic) class
    
    metrics = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1-Score": f1_score(y, y_pred),
        "Log-Loss": log_loss(y, y_prob),
        "AUROC": roc_auc_score(y, y_prob),
        "AUPRC": average_precision_score(y, y_prob)
    }
    
    print(f"\n===== {set_name.upper()} PERFORMANCE =====")
    for metric, value in metrics.items():
        print(f"{metric:12}: {value:.4f}")
    
    return metrics

In [2]:
# 1. Load train / validation / test datasets
dataset_train = pd.read_csv("../data/dataset_train.csv")
dataset_val   = pd.read_csv("../data/dataset_val.csv")
dataset_test  = pd.read_csv("../data/dataset_test.csv")

# 2. Preprocess (Drop non-feature columns)
DROP_COLS = ["index", "peptide", "HLA", "hla_sequence"]
TARGET_COL = "Label"

# Split labels
y_train = dataset_train[TARGET_COL].values
y_val   = dataset_val[TARGET_COL].values
y_test  = dataset_test[TARGET_COL].values

# Split features
X_train = dataset_train.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_train.columns])
X_val   = dataset_val.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_val.columns])
X_test  = dataset_test.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_test.columns])

# 3. Combine for consistent encoding
X_all = pd.concat(
    [X_train, X_val, X_test],
    axis=0,
    keys=["train", "val", "test"]
)

# 4. Encode categorical columns
cat_cols = X_all.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Encoding categorical columns: {cat_cols}")

X_all = pd.get_dummies(X_all, columns=cat_cols)

# 5. Split back
X_train = X_all.loc["train"]
X_val   = X_all.loc["val"]
X_test  = X_all.loc["test"]

print(f"Train: {X_train.shape}, Classes: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val:   {X_val.shape}, Classes: {dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"Test:  {X_test.shape}, Classes: {dict(zip(*np.unique(y_test, return_counts=True)))}")

# 6. Initialize and Fit Random Forest
rf_model = RandomForestFromScratch(n_estimators=50, max_depth=10)
rf_model.fit(X_train, y_train)

# 7. Performance Assessment
train_results = run_performance_assessment(rf_model, X_train, y_train, "Training Set")
val_results   = run_performance_assessment(rf_model, X_val, y_val, "Validation Set")
test_results  = run_performance_assessment(rf_model, X_test, y_test, "Test Set")

Encoding categorical columns: []
Train: (5707, 290), Classes: {np.int64(0): np.int64(3143), np.int64(1): np.int64(2564)}
Val:   (1446, 290), Classes: {np.int64(0): np.int64(778), np.int64(1): np.int64(668)}
Test:  (1815, 290), Classes: {np.int64(0): np.int64(991), np.int64(1): np.int64(824)}

===== TRAINING SET PERFORMANCE =====
Accuracy    : 0.8821
Precision   : 0.8405
Recall      : 0.9103
F1-Score    : 0.8740
Log-Loss    : 0.2995
AUROC       : 0.9623
AUPRC       : 0.9577

===== VALIDATION SET PERFORMANCE =====
Accuracy    : 0.7801
Precision   : 0.7391
Recall      : 0.8099
F1-Score    : 0.7729
Log-Loss    : 0.7030
AUROC       : 0.8578
AUPRC       : 0.8240

===== TEST SET PERFORMANCE =====
Accuracy    : 0.7912
Precision   : 0.7385
Recall      : 0.8362
F1-Score    : 0.7843
Log-Loss    : 0.6789
AUROC       : 0.8736
AUPRC       : 0.8247


In [3]:
from utils import save_model
OUTPUT_DIR = "../models"

/home/hamdaalhosani/Downloads/yes/envs/benchmark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "random_forest_scratch_v1"

metadata = {
    "dataset": "dataset_train / dataset_val / dataset_test",
    "features_shape": X_train.shape,
    "features": X_train.columns.tolist(),
    "n_estimators": rf_model.n_estimators,
    "max_depth": rf_model.max_depth,
    "min_samples_split": rf_model.min_samples_split,
    "min_samples_leaf": rf_model.min_samples_leaf,
    "val_metrics": val_results,
    "test_metrics": test_results,
    "notes": "Custom RandomForestFromScratch implementation"
}

model_path = save_model(
    model=rf_model,
    output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME,
    metadata=metadata
)

Saved model: ../models/random_forest_scratch_v1.pkl
Saved metadata: ../models/random_forest_scratch_v1_metadata.json


In [5]:
# random search for RandomForestFromScratch

configs = [
    {"n_estimators": 25,  "max_depth": 5,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 50,  "max_depth": 5,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 50,  "max_depth": 10, "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2,  "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 15, "min_samples_split": 5,  "min_samples_leaf": 2},
    {"n_estimators": 150, "max_depth": 15, "min_samples_split": 5,  "min_samples_leaf": 2},
]

In [6]:
results = []
best_model = None
best_score = -1
best_config = None

for i, cfg in enumerate(configs, start=1):
    version = f"v{i}"

    print(f"\nTraining random_forest_scratch_{version}: {cfg}")

    rf_model = RandomForestFromScratch(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        min_samples_split=cfg["min_samples_split"],
        min_samples_leaf=cfg["min_samples_leaf"]
    )

    rf_model.fit(X_train, y_train)

    y_val_pred = rf_model.predict(X_val)
    y_val_prob = rf_model.predict_proba(X_val)[:, 1]

    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_prob)
    val_pr_auc = average_precision_score(y_val, y_val_prob)
    val_f1 = f1_score(y_val, y_val_pred)

    print(f"Accuracy: {val_accuracy:.4f}")
    print(f"ROC AUC:  {val_auc:.4f}")
    print(f"PR AUC:   {val_pr_auc:.4f}")
    print(f"F1:       {val_f1:.4f}")

    save_model(
        model=rf_model,
        output_dir="../models",
        model_name=f"random_forest_scratch_{version}",
        metadata={
            "dataset": "dataset_train / dataset_val / dataset_test",
            "features_shape": X_train.shape,
            "features": X_train.columns.tolist(),
            "config": cfg,
            "val_accuracy": val_accuracy,
            "val_auc": val_auc,
            "val_pr_auc": val_pr_auc,
            "val_f1": val_f1,
            "notes": "RandomForestFromScratch hyperparameter search model"
        }
    )

    results.append({
        "version": version,
        **cfg,
        "val_accuracy": val_accuracy,
        "val_auc": val_auc,
        "val_pr_auc": val_pr_auc,
        "val_f1": val_f1
    })

    if val_auc > best_score:
        best_score = val_auc
        best_model = rf_model
        best_config = cfg

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_auc", ascending=False)

display(results_df)

print("\nBest RandomForestFromScratch config:")
print(best_config)
print(f"Best validation AUC: {best_score:.4f}")


Training random_forest_scratch_v1: {'n_estimators': 25, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Accuracy: 0.7566
ROC AUC:  0.8158
PR AUC:   0.7344
F1:       0.7545
Saved model: ../models/random_forest_scratch_v1.pkl
Saved metadata: ../models/random_forest_scratch_v1_metadata.json

Training random_forest_scratch_v2: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Accuracy: 0.7573
ROC AUC:  0.8231
PR AUC:   0.7586
F1:       0.7495
Saved model: ../models/random_forest_scratch_v2.pkl
Saved metadata: ../models/random_forest_scratch_v2_metadata.json

Training random_forest_scratch_v3: {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Accuracy: 0.7787
ROC AUC:  0.8575
PR AUC:   0.8254
F1:       0.7701
Saved model: ../models/random_forest_scratch_v3.pkl
Saved metadata: ../models/random_forest_scratch_v3_metadata.json

Training random_forest_scratch_v4: {'n_estimators': 100, 'max_depth': 10, 'min_samp

,version,n_estimators,max_depth,min_samples_split,min_samples_leaf,val_accuracy,val_auc,val_pr_auc,val_f1
5,v6,150,15,5,2,0.789765,0.870868,0.848916,0.778748
4,v5,100,15,5,2,0.782158,0.869004,0.845366,0.770241
3,v4,100,10,2,1,0.775934,0.859316,0.832186,0.767575
2,v3,50,10,2,1,0.778700,0.857505,0.825360,0.770115
1,v2,50,5,2,1,0.757261,0.823061,0.758638,0.749465
0,v1,25,5,2,1,0.756570,0.815751,0.734401,0.754533



Best RandomForestFromScratch config:
{'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2}
Best validation AUC: 0.8709
